<a href="https://colab.research.google.com/github/shreyansh8h/hello-world/blob/master/jul3_masterclass_RAG_Demo_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📄 RAG - Question Answering



In [5]:
!pip -q install gradio sentence-transformers faiss-cpu transformers accelerate bitsandbytes pypdf

In [6]:
from huggingface_hub import login

login("Replace HF Key")

In [7]:
import gradio as gr
import faiss
import torch
import re
import numpy as np
import time
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, TextIteratorStreamer
from pypdf import PdfReader
from huggingface_hub import login
import getpass
from google.colab import userdata
import threading

# Securely get Hugging Face token from Colab secrets
#try:
    #hf_token = "f_XVdeprOUXPNHcDDUkHtUAYiZFkYRguLHct"
    #login(hf_token)
#except:
    #print("Hugging Face token not found in Colab secrets. Please add it as 'HF_TOKEN'.")

## 1 · Sample Paragraph

In [8]:
SAMPLE_PARAGRAPHS = {

"Artificial Intelligence Today": """
Artificial intelligence has evolved from a narrow academic discipline into one of the most transformative technological forces of the 21st century. What began as early symbolic reasoning systems in the mid-20th century has grown into a sophisticated ecosystem of machine learning, deep neural networks, reinforcement learning, and multimodal foundation models that now power applications used by billions of people daily. AI is no longer confined to laboratories; it is embedded in smartphones, hospitals, financial markets, manufacturing plants, logistics networks, and national security systems.

At its core, modern AI is driven by machine learning, particularly deep learning architectures that learn patterns from vast amounts of data. Neural networks with millions or even billions of parameters are trained on diverse datasets to recognize speech, classify images, generate natural language, and make predictions. The emergence of transformer architectures marked a turning point, enabling models to understand context at scale and significantly improving performance in language translation, summarization, and conversational systems.

Large language models have demonstrated remarkable abilities in reasoning, coding assistance, content generation, and educational tutoring. These systems can draft legal documents, explain scientific theories, generate software prototypes, and provide personalized learning support. Multimodal systems now combine text, images, and audio inputs, moving closer to general-purpose digital assistants capable of interacting with the world in richer ways.

AI has reshaped healthcare through diagnostic imaging analysis, predictive analytics for disease outbreaks, and drug discovery acceleration. Algorithms can detect certain cancers from radiology scans with accuracy comparable to expert clinicians. In pharmaceuticals, AI-driven simulations significantly reduce the time required to identify promising molecular compounds. Personalized medicine, powered by genomic analysis and predictive modeling, is becoming increasingly feasible.

In finance, AI systems monitor transactions in real time to detect fraud, assess credit risk, and automate trading strategies. High-frequency trading algorithms execute decisions within microseconds, while risk modeling systems analyze complex global markets to predict volatility. Retail platforms use recommendation systems to personalize user experiences, increasing engagement and operational efficiency.

Autonomous systems represent another frontier. Self-driving vehicles rely on computer vision, sensor fusion, and reinforcement learning to navigate complex urban environments. Industrial robotics, powered by AI, operate in warehouses and manufacturing facilities with increasing autonomy. Drones assist in agriculture, surveying, and disaster response.

Despite these advancements, AI development raises significant ethical and societal concerns. Algorithmic bias can reinforce systemic inequalities when training data reflects historical discrimination. Facial recognition systems have shown disparities in accuracy across demographic groups. Addressing fairness requires diverse datasets, transparency in model design, and robust evaluation frameworks.

Data privacy remains a central issue. AI systems depend on massive datasets, often containing personal information. Ensuring that user data is protected while maintaining model performance is a complex technical and regulatory challenge. Techniques such as federated learning and differential privacy aim to mitigate risks.

The labor market is undergoing transformation as automation expands. While AI creates new job categories in data science and system engineering, it also threatens roles involving routine cognitive or physical tasks. Economists debate whether AI will augment human workers or displace them at scale. Reskilling initiatives and adaptive education systems are increasingly necessary.

AI governance has become a global priority. Governments and international organizations are drafting regulatory frameworks to ensure safety, accountability, and transparency. Discussions include mandatory auditing, explainability requirements, and liability standards for AI-driven decisions. Balancing innovation with regulation remains a delicate task.

Environmental considerations are also emerging. Training large AI models requires substantial computational power, leading to increased energy consumption. Researchers are exploring more efficient algorithms, specialized hardware accelerators, and renewable-powered data centers to reduce carbon footprints.

The future of AI may involve more collaborative human-AI systems rather than fully autonomous replacements. Augmented intelligence frameworks emphasize partnership, where AI handles data-heavy analysis while humans provide judgment, creativity, and ethical oversight. This hybrid approach could maximize benefits while minimizing harm.

Ultimately, artificial intelligence represents both extraordinary promise and profound responsibility. Its trajectory will shape economies, geopolitics, education systems, and even cultural norms. The challenge ahead is ensuring that AI development aligns with human values, promotes equity, and contributes to sustainable global progress.
""",

"Neuroscience Breakthroughs": """
Neuroscience has entered a transformative era, driven by technological innovation and interdisciplinary collaboration. The human brain, consisting of approximately 86 billion neurons interconnected by trillions of synapses, remains one of the most complex systems known to science. For decades, understanding its structure and function seemed nearly impossible, yet recent breakthroughs are steadily unraveling its mysteries.

Advances in neuroimaging technologies such as functional magnetic resonance imaging (fMRI), diffusion tensor imaging (DTI), and magnetoencephalography (MEG) allow researchers to observe brain activity in real time. These tools reveal how networks of neurons coordinate to support language, emotion, perception, and decision-making. Scientists can now track how information flows between regions during cognitive tasks.

The development of brain-computer interfaces (BCIs) marks a significant milestone. By decoding neural signals, BCIs enable paralyzed individuals to control prosthetic limbs or communicate via digital interfaces. Implanted electrode arrays capture electrical activity, which machine learning algorithms translate into actionable commands. This technology offers hope for restoring mobility and independence.

Deep brain stimulation (DBS) has proven effective for treating Parkinson’s disease, essential tremor, and certain psychiatric disorders. By delivering targeted electrical impulses to specific brain regions, DBS can modulate dysfunctional circuits. Ongoing research explores its potential for depression, obsessive-compulsive disorder, and chronic pain management.

Optogenetics has revolutionized experimental neuroscience. By genetically modifying neurons to respond to light, researchers can activate or silence specific cell populations with precision. This method provides insights into how neural circuits govern behavior, memory formation, and emotional regulation.

Large-scale initiatives aim to map the brain’s connectome — the comprehensive wiring diagram of neural connections. Understanding structural connectivity may illuminate how disorders such as schizophrenia and autism arise from disrupted communication between brain regions. High-resolution microscopy and computational modeling play essential roles in this effort.

Neuroplasticity research demonstrates that the brain remains adaptable throughout life. Learning new skills, recovering from injury, and adapting to environmental changes all involve synaptic remodeling. Rehabilitation programs increasingly leverage this plasticity to improve outcomes after strokes or traumatic brain injuries.

Artificial intelligence is becoming an essential tool in neuroscience research. Machine learning algorithms analyze complex neural datasets, identifying patterns that would be impossible to detect manually. Computational models simulate neural networks to test hypotheses about cognition and disease mechanisms.

Ethical considerations are expanding alongside technical capability. As scientists gain the ability to decode thoughts or influence neural activity, concerns about cognitive liberty and mental privacy intensify. Questions arise about consent, enhancement technologies, and equitable access to neurotechnological treatments.

The integration of neuroscience with psychology, computer science, and engineering is fostering a more holistic understanding of consciousness and behavior. Some researchers explore the neural correlates of subjective experience, seeking biological explanations for awareness itself.

Neuroscience also plays a crucial role in addressing global health challenges. Alzheimer’s disease and other neurodegenerative conditions are increasing with aging populations. Early detection biomarkers and targeted therapies remain urgent research priorities.

In education, insights into learning mechanisms inform teaching strategies that align with cognitive development principles. Understanding attention, memory consolidation, and emotional engagement enhances curriculum design.

As neuroscience advances, the boundary between therapy and enhancement becomes increasingly blurred. Memory augmentation, mood regulation implants, and cognitive optimization tools may become technically feasible. Society must determine appropriate ethical frameworks to guide such developments.

The coming decades promise deeper integration between neural technology and human life. Whether through medical intervention, cognitive augmentation, or fundamental research into consciousness, neuroscience will continue reshaping our understanding of what it means to be human.
""",

"Renewable Energy Revolution": """
The global energy system is undergoing a structural transformation driven by climate concerns, technological innovation, and economic opportunity. Renewable energy sources, once considered niche alternatives, are now central pillars of national energy strategies. Solar, wind, hydroelectric, geothermal, and bioenergy technologies are expanding at unprecedented rates.

Solar photovoltaic technology has experienced dramatic cost declines over the past two decades. Improvements in panel efficiency, manufacturing scale, and supply chain optimization have made solar one of the cheapest electricity sources worldwide. Large-scale solar farms now power millions of homes, while rooftop installations empower individual consumers.

Wind energy has similarly advanced. Modern turbines are taller, more efficient, and capable of generating electricity even at lower wind speeds. Offshore wind farms harness stronger and more consistent coastal winds, significantly increasing capacity factors. Floating wind platforms extend possibilities into deeper waters.

Energy storage represents a crucial component of renewable integration. Lithium-ion battery costs have fallen substantially, enabling grid-scale storage projects that stabilize supply fluctuations. Emerging technologies such as solid-state batteries, flow batteries, and green hydrogen storage aim to further enhance reliability.

Smart grids equipped with digital sensors and AI-driven analytics improve distribution efficiency. Real-time demand forecasting balances supply with consumption, minimizing waste and blackouts. Distributed energy resources allow communities to generate and store power locally.

Electrification of transportation complements renewable expansion. Electric vehicles reduce dependence on fossil fuels and can function as mobile storage units through vehicle-to-grid technologies. Charging infrastructure development remains essential for large-scale adoption.

Green hydrogen is gaining attention as a solution for decarbonizing heavy industry and long-haul transport. Produced via electrolysis using renewable electricity, hydrogen can replace coal in steelmaking and serve as fuel for shipping and aviation.

Policy frameworks significantly influence renewable deployment. Carbon pricing mechanisms, renewable portfolio standards, and investment tax credits incentivize clean energy adoption. International agreements encourage cooperation toward emission reduction targets.

Developing economies face unique challenges and opportunities. Renewable installations can expand electricity access in remote regions without extensive grid infrastructure. Decentralized solar microgrids provide reliable power for schools, hospitals, and businesses.

Despite progress, fossil fuels still dominate global energy consumption. Transitioning requires not only technological innovation but also infrastructure redesign, workforce retraining, and financial investment at unprecedented scales. Grid modernization, transmission expansion, and energy market reforms are critical components.

Environmental considerations extend beyond emissions. Responsible sourcing of battery materials, recycling of solar panels, and protection of ecosystems near wind farms require careful planning. Lifecycle assessments ensure sustainability across supply chains.

Economic implications are substantial. Renewable industries generate employment in manufacturing, installation, and maintenance. Countries investing early in clean energy technologies may gain competitive advantages in emerging markets.

The renewable energy revolution represents more than a technological shift; it is a societal transformation affecting geopolitics, economics, and environmental stewardship. Achieving a carbon-neutral future demands coordinated global action, sustained innovation, and equitable access to clean energy resources.

The trajectory of renewable energy will determine whether humanity successfully mitigates climate change while fostering sustainable development. Continued progress depends on integrating technology, policy, and public engagement into a coherent long-term strategy.
"""
}

In [9]:
def clean_text(text: str) -> str:
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def smart_chunk(text: str, chunk_size: int = 500, overlap_sentences: int = 2) -> list[str]:
    sentences = re.split(r'(?<=[.!?])\s+|\n\n+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []
    current_chunk_sentences = []
    current_chunk_length = 0

    for i, sentence in enumerate(sentences):
        sentence_length = len(sentence) + 1

        if current_chunk_length + sentence_length <= chunk_size:
            current_chunk_sentences.append(sentence)
            current_chunk_length += sentence_length
        else:
            if current_chunk_sentences:
                chunks.append(" ".join(current_chunk_sentences))


            overlap_start_index = max(0, i - overlap_sentences)
            current_chunk_sentences = list(sentences[overlap_start_index:i])
            current_chunk_length = sum(len(s) + 1 for s in current_chunk_sentences) if current_chunk_sentences else 0

            # Add the current sentence that didn't fit in the previous chunk
            current_chunk_sentences.append(sentence)
            current_chunk_length += sentence_length

    if current_chunk_sentences:
        chunks.append(" ".join(current_chunk_sentences))


    return [c for c in chunks if len(c) >= 20]

## 2 · Embedder & FAISS index

In [10]:
from functools import lru_cache

embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Embedder ready")


def build_index(chunks: list[str]):
    embeddings = embedder.encode(
        chunks,
        batch_size=64,
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)

    dim = embeddings.shape[1]
    n = len(embeddings)

    # IVF needs at least 39 vectors per cluster to be stable
    if n >= 256:
        nlist = min(100, n // 39)
        quantizer = faiss.IndexFlatIP(dim)
        index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
        index.train(embeddings)
        index.nprobe = min(10, nlist)
    else:
        # Small corpus — flat exact search is fastest
        index = faiss.IndexFlatIP(dim)

    index.add(embeddings)
    return index, embeddings

@lru_cache(maxsize=256)
def encode_query(query: str) -> np.ndarray:
    emb = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    return emb


def retrieve(query: str, chunks: list[str], index, k: int = 6):
    q_emb = encode_query(query)
    scores, ids = index.search(q_emb, k)
    return [(float(scores[0][i]), chunks[ids[0][i]]) for i in range(len(ids[0]))]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedder ready


## 3 · Model registry & loader

In [11]:
MODEL_NAMES = {
    "GPT2": "openai-community/gpt2",
    "Qwen 2.5 0.5B": "Qwen/Qwen2.5-0.5B",
    "Qwen 3 0.6B": "Qwen/Qwen3-0.6B",
    "Liquid LFM2 700M": "LiquidAI/LFM2-700M",
    "Liquid LFM2 350M": "LiquidAI/LFM2-350M"
}

In [12]:
MODEL_CONTEXT = {
    "GPT2": 1024,
    "Qwen 2.5 0.5B": 32768,
    "Qwen 3 0.6B": 32768,
    "Liquid LFM2 700M": 4096,
    "Liquid LFM2 350M": 4096,
}

In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
loaded_models = {}

# Using 4-bit quantization for better performance/memory trade-off
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

def load_model(name):
    if name in loaded_models:
        return loaded_models[name]

    model_id = MODEL_NAMES[name]
    print(f"Loading {name}...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

        # Ensure tokenizer has a pad_token
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config if device == "cuda" else None,
            device_map="auto" if device == "cuda" else None,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            trust_remote_code=True,
            low_cpu_mem_usage=True
        )

        pipe = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer
        )
        loaded_models[name] = pipe
        return pipe
    except Exception as e:
        print(f"Failed to load {name}: {e}")
        return None

In [14]:
print("Attempting to load 'Liquid LFM2 700M' model...")
test_model = load_model("Liquid LFM2 700M")
if test_model:
    print("✅ Model 'Liquid LFM2 700M' loaded successfully!")
else:
    print("❌ Failed to load model 'Liquid LFM2 700M'. Check previous error messages.")

Attempting to load 'Liquid LFM2 700M' model...
Loading Liquid LFM2 700M...


config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/91.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.73M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.48G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

✅ Model 'Liquid LFM2 700M' loaded successfully!


## 4 · Prompt builder & answer cleaner

In [15]:
def build_prompt(context: str, question: str) -> str:
    return (
        "You are a precise document question-answering assistant. Your goal is to provide accurate answers strictly based on the provided context.\n"
        "Answer ONLY using the information inside <context>. Do NOT invent information or use external knowledge. "
        "If the answer is not directly present or derivable from the context, reply exactly: Not found in document.\n\n"
        "<context>\n"
        f"{context}\n"
        "</context>\n\n"
        f"<question>{question}</question>\n\n"
        "Answer:"
    )


def clean_output(raw: str) -> str:

    if "Answer:" in raw:
        raw = raw.split("Answer:")[-1]

    raw = re.sub(r"[ \t]+", " ", raw)
    raw = re.sub(r"\n{2,}", "\n", raw)
    raw = raw.strip()

    match = re.search(r'[.!?][^.!?]*$', raw)
    if match and match.start() > len(raw) // 2:
        raw = raw[:match.start() + 1]

    return raw.strip()

In [16]:
VECTOR = None
CHUNKS = None


def ask(question, model_names, top_k=5, min_new_tokens=64, max_new_tokens=256):

    if VECTOR is None or CHUNKS is None:
        return "⚠️ Please select a paragraph first."

    scored_chunks = retrieve(question, CHUNKS, VECTOR, k=top_k)

    chunk_report = "\n".join(
        f"  **Chunk {i+1}** (score {s:.3f}): {c[:120]}…"
        for i, (s, c) in enumerate(scored_chunks)
    )

    context = " ".join(c for _, c in scored_chunks)
    prompt = build_prompt(context, question)
    answers = [f"### 🔍 Retrieved Chunks\n{chunk_report}\n"]

    for name in model_names:
        start_time = time.time()

        pipe = load_model(name)
        if pipe is None:
            continue

        tokenizer = pipe.tokenizer
        context_window = MODEL_CONTEXT[name]

        input_tokens = len(tokenizer(prompt)["input_ids"])
        # The half_window is a heuristic. We'll prioritize available_tokens and user's max_new_tokens.
        available_tokens = context_window - input_tokens

        if available_tokens <= 0:
            continue

        # Use user-specified min_new_tokens, ensuring it's not negative
        final_min = max(0, min_new_tokens)
        # Use user-specified max_new_tokens, capped by the actual available context tokens
        final_max = min(max_new_tokens, available_tokens)

        # Ensure final_min doesn't exceed final_max
        if final_min > final_max:
            final_min = final_max

        # --- Streaming text generation ---
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

        generation_kwargs = dict(
            input_ids=input_ids,
            streamer=streamer,
            max_new_tokens=final_max,
            min_new_tokens=final_min,
            do_sample=True,
            temperature=0.5,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

        thread = threading.Thread(target=pipe.model.generate, kwargs=generation_kwargs)
        thread.start()

        full_generated_text = ""
        for token in streamer:
            full_generated_text += token
        # --- End streaming text generation ---

        text = clean_output(full_generated_text)
        end_time = time.time()
        execution_time = end_time - start_time
        answers.append(f"### 🤖 {name} (Time: {execution_time:.2f}s)\n\n{text}")

    return "\n\n---\n\n".join(answers)

In [17]:
def index_paragraph(selection):
    global VECTOR, CHUNKS
    global ALL_INDEXES, ALL_CHUNKS # Declare these as global

    if selection in ALL_INDEXES and selection in ALL_CHUNKS:
        VECTOR = ALL_INDEXES[selection]
        CHUNKS = ALL_CHUNKS[selection]
        return f"✅ Loaded {len(CHUNKS)} chunks from '{selection}'"
    else:
        # Fallback for unexpected cases or if ALL_INDEXES/ALL_CHUNKS are not populated
        text = clean_text(SAMPLE_PARAGRAPHS[selection])
        CHUNKS = smart_chunk(text)
        VECTOR, _ = build_index(CHUNKS)
        return f"Indexed {len(CHUNKS)} chunks from '{selection}' (freshly built)"

## 5 · Gradio UI

In [ ]:
import time

# Initialize globals if not present
ALL_INDEXES = {}
ALL_CHUNKS = {}

print("Pre-indexing all sample paragraphs...")
for name, text in SAMPLE_PARAGRAPHS.items():
    if name not in ALL_INDEXES:
        cleaned = clean_text(text)
        chunks = smart_chunk(cleaned)
        index, _ = build_index(chunks)
        ALL_INDEXES[name] = index
        ALL_CHUNKS[name] = chunks
print("✅ All paragraphs pre-indexed.")

# Pre-load all models to avoid latency during first query
print("Pre-loading models...")
for name in MODEL_NAMES:
    load_model(name)
print("✅ All models loaded.")

with gr.Blocks(title="RAG") as demo:
    gr.Markdown("# 📄 RAG — Q&A")
    gr.Markdown("Select a paragraph → It auto-indexes → Ask your question")

    with gr.Row():
        with gr.Column(scale=1):
            paragraph_select = gr.Dropdown(
                choices=list(SAMPLE_PARAGRAPHS.keys()),
                value="Artificial Intelligence Today",
                label="📚 Select Sample Paragraph"
            )
            index_status = gr.Markdown(f"✅ Loaded {len(ALL_CHUNKS['Artificial Intelligence Today'])} chunks.")

            paragraph_display = gr.Textbox(
                value=SAMPLE_PARAGRAPHS["Artificial Intelligence Today"],
                label="📖 Selected Paragraph",
                lines=8,
                interactive=False
            )
            gr.Markdown("---")
            model_select = gr.CheckboxGroup(
                list(MODEL_NAMES.keys()),
                value=["Liquid LFM2 700M"],
                label="🤖 Select Models"
            )
            top_k_slider = gr.Slider(2, 10, value=5, step=1, label="🔎 Top-K Retrieved Chunks")
            min_tokens_slider = gr.Slider(0, 512, value=0, step=16, label="🧾 Minimum Output Tokens")
            max_tokens_slider = gr.Slider(32, 1024, value=256, step=32, label="🧾 Maximum Output Tokens")

        with gr.Column(scale=2):
            question_box = gr.Textbox(label="💬 Your Question", lines=2)
            ask_btn = gr.Button("🚀 Ask", variant="primary")
            output_box = gr.Markdown(label="📊 Model Outputs")

    def handle_paragraph_change(selection):
        status = index_paragraph(selection)
        return status, SAMPLE_PARAGRAPHS[selection]

    paragraph_select.change(
        handle_paragraph_change,
        inputs=paragraph_select,
        outputs=[index_status, paragraph_display]
    )

    ask_btn.click(
        ask,
        inputs=[question_box, model_select, top_k_slider, min_tokens_slider, max_tokens_slider],
        outputs=output_box
    )

# Initialize state for current selection
index_paragraph("Artificial Intelligence Today")

demo.launch(share=True, debug=True)

Pre-indexing all sample paragraphs...
✅ All paragraphs pre-indexed.
Pre-loading models...
Loading GPT2...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/709M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

✅ All models loaded.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d36f55d6fca4949f27.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: 